In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as _sum, lag, to_date, date_format

spark = SparkSession.builder.appName("InstagramGrowth").getOrCreate()

# Sample Data
data = [
    ("U1", "2024-01-01", 100),
    ("U1", "2024-02-01", 200),
    ("U1", "2024-03-01", 300),
    ("U2", "2024-01-01", 500),
    ("U2", "2024-02-01", 400),
    ("U2", "2024-03-01", 600)
]

columns = ["user_id", "post_date", "likes"]

df = spark.createDataFrame(data, columns) \
          .withColumn("post_date", to_date("post_date"))

# Step 1: Extract month
df = df.withColumn("month", date_format("post_date", "yyyy-MM"))

# Step 2: Monthly aggregation
monthly_df = df.groupBy("user_id", "month") \
               .agg(_sum("likes").alias("monthly_likes"))

# Step 3: Window
window_spec = Window.partitionBy("user_id").orderBy("month")

# Step 4: Compare with previous months
result_df = monthly_df.withColumn("prev1", lag("monthly_likes", 1).over(window_spec)) \
    .withColumn("prev2", lag("monthly_likes", 2).over(window_spec)) \
    .filter(
        (col("monthly_likes") > col("prev1")) &
        (col("prev1") > col("prev2"))
    )

result_df.show()